# 3_Silver_Inventario
## Transformación del dataset Inventario: Bronze → Silver

**Objetivo:** leer `inventario` desde Bronze, validar integridad referencial contra los productos de Bronze, separar registros inválidos y preparar la tabla Silver.

**Reglas del proceso**
- Se conserva Bronze sin modificaciones.
- La clave de negocio es compuesta: `fecha_corte`, `producto_id`, `almacen_id`.
- Se rechazan registros con campos obligatorios nulos.
- Se rechazan registros cuyo `producto_id` no exista en `bronze.productos`.
- No se reemplazan valores ni se eliminan registros sin trazabilidad.
- Los registros rechazados se almacenan en una tabla independiente de Silver.

## 1. Inicialización

In [ ]:
dbutils.widgets.removeAll()

In [ ]:
from pyspark.sql.functions import col, trim, lit, current_timestamp
from pyspark.sql.types import DateType, IntegerType

## 2. Parámetros

In [ ]:
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquemaBronze = "bronze"
esquemaSilver = "silver"

ruta_silver = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/silver/"

tabla_inventario_bronze = f"{catalogo}.{esquemaBronze}.inventario"
tabla_productos_bronze = f"{catalogo}.{esquemaBronze}.productos"
tabla_inventario_silver = f"{catalogo}.{esquemaSilver}.inventario"
tabla_inventario_rechazados = f"{catalogo}.{esquemaSilver}.inventario_rechazados"

print("Catálogo:", catalogo)
print("Origen:", tabla_inventario_bronze)
print("Referencia:", tabla_productos_bronze)
print("Destino:", tabla_inventario_silver)
print("Rechazados:", tabla_inventario_rechazados)
print("Ruta Silver:", ruta_silver)

## 3. Lectura desde Bronze

In [ ]:
df_inventario_bronze = spark.table(tabla_inventario_bronze)
df_productos_bronze = spark.table(tabla_productos_bronze)

print("Registros en Inventario Bronze:", df_inventario_bronze.count())
df_inventario_bronze.printSchema()

## 4. Separación de registros inválidos

Se consideran inválidos:
1. Registros con `fecha_corte`, `producto_id`, `almacen_id`, `stock_disponible` o `stock_minimo` nulos.
2. Registros cuyo `producto_id` no existe en `bronze.productos`.

La referencia se valida contra **Bronze de productos**, no contra Silver, porque la validación de existencia de la clave foránea debe ser independiente de las reglas de calidad aplicadas al costo del producto.

In [ ]:
productos_validos = (
    df_productos_bronze
    .select("producto_id")
    .distinct()
)

condicion_campos_nulos = (
    col("fecha_corte").isNull() |
    col("producto_id").isNull() |
    col("almacen_id").isNull() |
    col("stock_disponible").isNull() |
    col("stock_minimo").isNull()
)

df_inventario_rechazados_nulos = (
    df_inventario_bronze
    .filter(condicion_campos_nulos)
    .withColumn("motivo_rechazo", lit("CAMPOS_OBLIGATORIOS_NULOS"))
    .withColumn("fecha_proceso", current_timestamp())
)

df_inventario_sin_nulos = df_inventario_bronze.filter(~condicion_campos_nulos)

df_inventario_rechazados_producto = (
    df_inventario_sin_nulos
    .join(productos_validos, on="producto_id", how="left_anti")
    .withColumn("motivo_rechazo", lit("PRODUCTO_NO_EXISTE"))
    .withColumn("fecha_proceso", current_timestamp())
)

df_inventario_candidatos = (
    df_inventario_sin_nulos
    .join(productos_validos, on="producto_id", how="inner")
)

print("Inventario Bronze:", df_inventario_bronze.count())
print("Rechazados por nulos:", df_inventario_rechazados_nulos.count())
print("Rechazados por producto inexistente:", df_inventario_rechazados_producto.count())
print("Candidatos Silver:", df_inventario_candidatos.count())

## 5. Validación de duplicados

In [ ]:
clave_inventario = ["fecha_corte", "producto_id", "almacen_id"]

duplicados_inventario = (
    df_inventario_candidatos
    .groupBy(*clave_inventario)
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Grupos de claves duplicadas:", duplicados_inventario)

assert duplicados_inventario == 0, "Se encontraron claves compuestas duplicadas." 

## 6. Preparación de registros Silver

In [ ]:
df_inventario_silver = (
    df_inventario_candidatos
    .select(
        "fecha_corte",
        "producto_id",
        "almacen_id",
        "stock_disponible",
        "stock_minimo",
        "_source_file",
        "_ingestion_timestamp"
    )
)

print("Registros preparados para Silver:", df_inventario_silver.count())
df_inventario_silver.printSchema()

## 7. Validación de conteos y calidad

In [ ]:
conteo_bronze = df_inventario_bronze.count()
conteo_silver = df_inventario_silver.count()
conteo_rechazados_nulos = df_inventario_rechazados_nulos.count()
conteo_rechazados_producto = df_inventario_rechazados_producto.count()
conteo_rechazados = conteo_rechazados_nulos + conteo_rechazados_producto

print("Bronze:", conteo_bronze)
print("Silver:", conteo_silver)
print("Rechazados por nulos:", conteo_rechazados_nulos)
print("Rechazados por producto inexistente:", conteo_rechazados_producto)
print("Total rechazados:", conteo_rechazados)
print("Conciliación:", conteo_silver + conteo_rechazados)

assert conteo_bronze == conteo_silver + conteo_rechazados, \
    "Los conteos de Bronze, Silver y rechazados no concilian."

assert duplicados_inventario == 0, \
    "Se encontraron claves compuestas duplicadas."

print("Validaciones de conteos correctas.")

## 8. Inspección antes de escribir

In [ ]:
display(df_inventario_silver.orderBy("fecha_corte", "producto_id", "almacen_id").limit(20))

In [ ]:
display(df_inventario_rechazados_producto.orderBy("producto_id").limit(20))

## 9. Preparar esquema Silver

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquemaSilver}")
print(f"Esquema disponible: {catalogo}.{esquemaSilver}")

## 10. Escritura de tablas Delta

Ejecutar estas celdas solo después de revisar las validaciones y la inspección. `overwrite` reconstruye las tablas Silver si se vuelve a ejecutar el proceso. No modifica Bronze.

In [ ]:
df_inventario_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{ruta_silver}inventario/") \
    .saveAsTable(tabla_inventario_silver)

print("Tabla creada:", tabla_inventario_silver)

In [ ]:
df_inventario_rechazados = (
    df_inventario_rechazados_nulos
    .unionByName(df_inventario_rechazados_producto, allowMissingColumns=True)
)

df_inventario_rechazados.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{ruta_silver}inventario_rechazados/") \
    .saveAsTable(tabla_inventario_rechazados)

print("Tabla creada:", tabla_inventario_rechazados)

## 11. Validación final

In [ ]:
df_inventario_silver_tabla = spark.table(tabla_inventario_silver)
df_inventario_rechazados_tabla = spark.table(tabla_inventario_rechazados)

print("Inventario Silver:", df_inventario_silver_tabla.count())
print("Inventario rechazados:", df_inventario_rechazados_tabla.count())
df_inventario_silver_tabla.printSchema()

assert df_inventario_silver_tabla.count() + df_inventario_rechazados_tabla.count() == conteo_bronze, \
    "La conciliación final no coincide con Bronze."

print("Validación final correcta.")

## Resultado esperado

Según el diagnóstico inicial:
- Bronze inventario: **10,000 registros**.
- Claves compuestas duplicadas: **0**.
- Productos inexistentes referenciados por inventario: **20**.
- Se espera que esos registros queden en `silver.inventario_rechazados`.
- La cantidad final de Silver se confirmará durante la ejecución y conciliación.

**Importante:** la validación de integridad referencial compara contra `bronze.productos`. No se filtra contra `silver.productos`, porque allí se rechazaron productos por costo unitario nulo, una regla distinta de la existencia de la clave foránea.